# 00 — Reproducible Python Environments
## From 'works on my machine' to an experiment other people can re-run

**Rule:** every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you to complete.

You will see two kinds of prompt:

- **Predict** — before you run a cell, write down what you think it will print, and why.
- **What you just saw** — a short note after a cell that explains the output in plain words. Compare it to your prediction. A wrong prediction is useful: it shows you which idea to fix.

The project at the end has no single right answer. You get the goal and the acceptance criteria; the code is yours.

## 1. The environment is part of the result

When you run an AI experiment, the number you get out depends on more than your code and your data. It also depends on:

- the exact versions of the packages you installed (`numpy`, `torch`, ...),
- the Python interpreter itself,
- compiled libraries underneath Python (BLAS for matrix maths, CUDA for GPUs),
- the operating system,
- the random seeds.

If any of these changes without you noticing, your result can change without you noticing.

You cannot freeze all of it at once. You freeze it in **layers**, and each layer covers a bit more:

| What you use | What it pins | What it still does *not* pin |
|---|---|---|
| a virtual environment (`venv`, `uv`, conda) | which Python packages are installed | the Python version, compiled libraries, the OS |
| a **lock file** (`uv.lock`, `requirements.lock.txt`) | the one exact set of package versions that was resolved | compiled libraries outside the resolver, the OS |
| a container image (Docker) | the OS and system libraries too | the GPU driver, the hardware, timing effects |

![Layers of reproducibility](assets/environment_layers.svg)

**`requirements.txt` vs a lock file.** A `requirements.txt` usually says what you *want* ("some `numpy` 2.x"). A **lock file** records what you actually *got* — every package, including the ones pulled in indirectly, pinned to an exact version. A second machine that installs from the lock file ends up with the same set.

**Predict.** The next cell prints facts about the running Python. On a machine with **no** virtual environment active, which of these two will be equal: `prefix` and `base_prefix`? What changes once a `venv` is active?

In [1]:
import platform, sys
from typing import Sequence

def runtime_report() -> dict[str, str]:
    """The facts you would paste into a bug report or an experiment log."""
    return {
        "python": sys.version.split()[0],
        "implementation": platform.python_implementation(),
        "platform": platform.platform(),
        "executable": sys.executable,       # the Python binary running this notebook
        "prefix": sys.prefix,               # root of the environment that is active now
        "base_prefix": sys.base_prefix,     # root of the Python that environment was built from
        "in_virtualenv": str(sys.prefix != sys.base_prefix),
    }

report = runtime_report()
for key, value in report.items():
    print(f"{key:15}: {value}")

python         : 3.13.5
implementation : CPython
platform       : Linux-6.12.105+deb13-amd64-x86_64-with-glibc2.41
executable     : /usr/bin/python3
prefix         : /usr
base_prefix    : /usr
in_virtualenv  : False


### What you just saw

`runtime_report()` returns a plain dictionary; the loop prints one line per fact so you can read it.

- **`executable`** is the single most useful line: the exact Python running this notebook. If it is not the Python you installed your packages into, that is your bug.
- **`base_prefix`** is where the "real" Python lives. **`prefix`** is the environment that is active right now.
  - No environment active → `prefix == base_prefix`, and `in_virtualenv` is `False`.
  - A `venv` / `uv` environment active → `prefix` points inside your project (`.../my-project/.venv`), while `base_prefix` still points at the system Python.
- **conda** is the exception: it does not always separate `prefix` and `base_prefix`. There, also look at `CONDA_PREFIX` and at `executable`.

## 2. Which tool for which job

| Your situation | Sensible default | What that tool controls |
|---|---|---|
| plain Python, no compiled extras | `venv` + `pip` | Python packages |
| you want a fast workflow and a lock file | `uv` | Python versions, environments, packages, lock file |
| you need CUDA or other compiled scientific libraries | conda-forge (Miniforge, mamba, micromamba) | Python **and** non-Python binaries |
| the OS itself must match | Docker | the whole runtime image |

The conda family has confusing names. In one line each: **Anaconda** is a big bundle; **Miniconda** is a small installer; **Miniforge** is a small installer set to the community `conda-forge` channel; **mamba** is a faster conda; **micromamba** is a single-file mamba. You need them when Python depends on compiled libraries — but they are not automatically "more reproducible" than `venv` or `uv`.

Rule of thumb:

1. Plain Python → `venv` + `pip`.
2. Want speed + Python-version control + a lock file → `uv`.
3. Need conda packages / compiled dependencies → Miniforge or micromamba.
4. Also need to pin the OS → add Docker on top.

If you must `pip install` inside a conda environment, install the conda packages first, then pip, and write down both lists.

### The commands, for reference

```bash
# venv + pip
python3 -m venv .venv
. .venv/bin/activate                       # Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m pip freeze > requirements.lock.txt

# uv
uv python install 3.12
uv init
uv add numpy
uv sync                                    # creates/updates uv.lock and the environment

# conda-forge with micromamba
micromamba create -n ai-course -c conda-forge python=3.12 numpy
micromamba activate ai-course
micromamba env export --from-history > environment.yml       # what you asked for
micromamba env export > environment.lock.yml                 # everything that got resolved
```

Running a command is not proof of anything. The proof is the report: interpreter path, Python version, package versions, platform, hardware assumptions, and the dependency list you built the environment from.

In [2]:
import os
from pathlib import Path

def environment_diagnosis() -> dict[str, str]:
    """Does the environment your *shell* thinks is active match the one the *kernel* runs?"""
    shell_env = os.environ.get("VIRTUAL_ENV") or os.environ.get("CONDA_PREFIX") or ""
    matches = bool(shell_env) and Path(shell_env).resolve() == Path(sys.prefix).resolve()
    return {
        "kernel_executable": sys.executable,
        "kernel_prefix": sys.prefix,
        "kernel_is_isolated": str(sys.prefix != sys.base_prefix),
        "shell_env_var": shell_env or "(none set)",
        "shell_matches_kernel": str(matches) if shell_env else "no shell variable to compare",
    }

for key, value in environment_diagnosis().items():
    print(f"{key:20}: {value}")

kernel_executable   : /usr/bin/python3
kernel_prefix       : /usr
kernel_is_isolated  : False
shell_env_var       : (none set)
shell_matches_kernel: no shell variable to compare


### What you just saw

The shell (your terminal) and the Jupyter kernel are two different things. Your terminal can be "in" one environment while the notebook runs a different Python.

- `os.environ` inside a notebook is the kernel's environment, frozen when the kernel started. It is **not** your live terminal.
- If you activate a `venv` and *then* start Jupyter from it, `VIRTUAL_ENV` is set and matches.
- If you start Jupyter first and pick a kernel from a menu, `VIRTUAL_ENV` can be empty even though `sys.prefix` is correct.
- So trust `sys.prefix` and `sys.executable`, not the environment variable.

## Common problems

| Symptom | Likely cause | How to check |
|---|---|---|
| `ModuleNotFoundError` right after a successful `pip install` | `pip` installed into a different Python | compare `python -m pip --version` with `sys.executable` |
| works in the terminal, fails in Jupyter | wrong kernel selected | compare `sys.executable` in both |
| install fails with a compiler / native-library error | no prebuilt package for this OS + Python | check platform, Python version, the package's docs |
| a result changes after reinstalling | a package you did not pin moved | use a lock file; write down versions |
| CPU and GPU give different numbers | different hardware or float precision | record the hardware and the numeric settings |

## 3. Reading a dependency line

A line in a requirements file can do four things:

| Form | Example | Meaning |
|---|---|---|
| exact pin | `numpy==2.2.4` | this exact version — used in lock files and applications |
| range | `numpy>=2.0,<3` | any version in this range — used by libraries, resolved later |
| environment marker | `torch==2.4.0 ; sys_platform == 'linux'` | only install this on some platforms |
| hash | `--hash=sha256:...` | the downloaded file must match this checksum |

`pip freeze` prints today's fully-resolved set. It is a fine starting point for a lock file, but it does **not** record hashes or markers, and it prints broken lines for local installs (`-e .`, `pkg @ file:///...`). For anything that matters, use a real lock tool (`uv lock`, `pip-compile`, `conda-lock`).

**Predict.** The next cell reads the text below and keeps only the exact-pin lines. Which of these six lines survive, and which are dropped?

```
# experiment A          <- comment
NumPy==2.2.4            <- exact pin
scikit_learn==1.6.1     <- exact pin
torch==2.4.0 ; ...      <- exact pin with a marker
mypackage @ file:///... <- local install
numpy>=2,<3             <- a range, not a pin
--index-url https://... <- an option
```

In [3]:
import re

# Matches:  name==version   (optionally followed by  "; marker")
PINNED = re.compile(r"^(?P<name>[A-Za-z0-9._-]+)==(?P<version>[^;\s]+)\s*(?:;.*)?$")

def parse_freeze(text: str) -> dict[str, str]:
    """Return {name: version} for the exact-pin lines only; skip everything else."""
    packages: dict[str, str] = {}
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith(("#", "-")) or " @ " in line:
            continue                                   # comment, option, or local install
        match = PINNED.match(line)
        if match:
            # lower-case and turn runs of . _ - into a single -  (part of PEP 503)
            name = re.sub(r"[-_.]+", "-", match["name"]).lower()
            packages[name] = match["version"]
    return packages

sample = """# experiment A
NumPy==2.2.4
scikit_learn==1.6.1
torch==2.4.0 ; sys_platform == 'linux'
mypackage @ file:///home/user/mypackage
numpy>=2,<3
--index-url https://example.invalid/simple
"""

result = parse_freeze(sample)
for name, ver in result.items():
    print(f"kept:    {name} == {ver}")
print("dropped: the comment, the local install, 'numpy>=2,<3' (a range), and the --index-url option")

assert result == {"numpy": "2.2.4", "scikit-learn": "1.6.1", "torch": "2.4.0"}, result

kept:    numpy == 2.2.4
kept:    scikit-learn == 1.6.1
kept:    torch == 2.4.0
dropped: the comment, the local install, 'numpy>=2,<3' (a range), and the --index-url option


### What you just saw

Three lines survived and were normalised:

- `NumPy` became `numpy` (lower-cased),
- `scikit_learn` became `scikit-learn` (the `_` became `-`),
- `torch==2.4.0 ; sys_platform == 'linux'` kept its version; the marker after `;` was ignored for this simple parser.

The range `numpy>=2,<3` was dropped on purpose: it is not one version, so a lock file cannot use it as-is. The `assert` at the end is the specification — if you change the function and it still passes, the behaviour is preserved.

## 4. A reproducibility checklist

Packages are only one axis. For someone else — or you in six months — to re-run an experiment, write down:

- **Code version** — the git commit, and whether the working copy was clean.
- **Environment** — the lock file, the Python version, how the environment was made.
- **Data** — a path *and* a content hash (or a dataset version). "The CSV from Slack" is not a version.
- **Configuration** — every setting and flag, including the defaults, saved to a file.
- **Seeds** — for `random`, `numpy`, your framework, and `PYTHONHASHSEED`. Note anything that is still random after that.
- **Hardware & numerics** — CPU / GPU model, driver version, float precision, any "deterministic mode" switch.
- **Environment report** — the output of `runtime_report()`.
- **Evaluation code** — the metric, stored next to the model.

And: keep secrets out of notebooks and out of git. Commit a `.env.example` that lists variable *names* with no values. In CI, rebuild from the lock file on a clean machine and run the notebooks end to end — that is the only honest reproducibility test.

In [4]:
from importlib.metadata import PackageNotFoundError, version

def package_versions(names: Sequence[str]) -> dict[str, str]:
    """The installed version of each package (or 'not installed')."""
    report: dict[str, str] = {}
    for name in names:
        try:
            report[name] = version(name)
        except PackageNotFoundError:
            report[name] = "not installed"      # you must handle this branch
    return report

# The name you pip-install is not always the name you import:
#   "scikit-learn" is imported as  sklearn
#   "Pillow"       is imported as  PIL
for name, ver in package_versions(["numpy", "matplotlib", "jupyterlab", "pydantic"]).items():
    print(f"{name:12}: {ver}")

numpy       : 2.2.4
matplotlib  : 3.10.1+dfsg1
jupyterlab  : not installed
pydantic    : not installed


### What you just saw

Whatever is not installed in this environment shows `not installed` instead of raising. A report that crashes on a missing package is not useful in a bug report, so the function handles that case explicitly.

Note the name mismatch: you `pip install scikit-learn` but you `import sklearn`. `version()` wants the install name.

## 5. Seeds, and what they do not fix

A seed makes a pseudo-random generator replay the same sequence. That covers data shuffling, weight initialisation, dropout, augmentation — **as long as every generator is seeded**.

A seed does **not** control:

- **Hash order** — the iteration order of sets, and some `dict`-based code. Fixed by the `PYTHONHASHSEED` environment variable, which must be set *before* Python starts (you cannot change it from a cell).
- **Thread / async order** — the order concurrent results come back in (notebook 05).
- **Floating-point addition order** — parallel sums can reorder additions, and `a + b + c` is not bit-for-bit the same as `a + (b + c)` in floats. Frameworks have a "deterministic mode" that trades speed for exact repeatability.
- **Library / driver versions** — a new BLAS or cuDNN can move results within rounding error.

**Predict.** In the next cell: which printed lines are identical on a re-run, and which change every time?

In [5]:
import random
import numpy as np

def seeded_draw(seed: int) -> dict[str, list[float]]:
    random.seed(seed)
    np.random.seed(seed)
    return {
        "stdlib": [round(random.random(), 4) for _ in range(3)],
        "numpy": [round(float(x), 4) for x in np.random.rand(3)],
    }

run_1 = seeded_draw(0)
run_2 = seeded_draw(0)
no_seed = [round(float(x), 4) for x in np.random.default_rng().random(3)]   # fresh randomness every call

print("seeded, run 1:", run_1)
print("seeded, run 2:", run_2)
print("run 1 == run 2:", run_1 == run_2, " <- same seed, so identical")
print("no seed       :", no_seed, " <- different every time you run this cell")
print("PYTHONHASHSEED:", os.environ.get("PYTHONHASHSEED", "(unset -> random at each start)"))

assert run_1 == run_2

seeded, run 1: {'stdlib': [0.8444, 0.758, 0.4206], 'numpy': [0.5488, 0.7152, 0.6028]}
seeded, run 2: {'stdlib': [0.8444, 0.758, 0.4206], 'numpy': [0.5488, 0.7152, 0.6028]}
run 1 == run 2: True  <- same seed, so identical
no seed       : [0.3522, 0.5565, 0.0996]  <- different every time you run this cell
PYTHONHASHSEED: (unset -> random at each start)


### What you just saw

`run_1` and `run_2` are equal because both called `seeded_draw(0)` — same seed, same sequence. `no_seed` uses a generator with no seed, so it draws fresh randomness and prints something different on every run.

If `PYTHONHASHSEED` prints `(unset ...)`, then set iteration order in this process is randomised. To pin it you must start Python with `PYTHONHASHSEED=0` in the environment — a cell cannot do it after the fact.

## Project — Reproducible experiment capsule

Build a small repository that a stranger can clone and run to reproduce a trivial "experiment" (for example: draw 1000 seeded samples and report their mean). The deliverable is the *capsule*, not the experiment.

**Required files**

- `pyproject.toml` — project metadata and dependencies
- a lock file — `uv.lock`, `requirements.lock.txt`, or equivalent
- `.gitignore` — excludes `.venv/`, `__pycache__/`, `.env`, generated outputs
- `.env.example` — names of any environment variables, no values
- `report.py` — implements `export_environment_report(path: str) -> None`
- `README.md` — install and run instructions, plus the CPU/GPU and data assumptions

**`pyproject.toml` starting point**

```toml
[project]
name = "experiment-capsule"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = ["numpy>=2.0,<3"]

[tool.experiment]
seed = 0
```

**`export_environment_report` must serialise as JSON** at least: Python version and executable, platform, `prefix` / `base_prefix`, the installed version of every declared dependency, the configured seed, and a UTC timestamp. Reuse `runtime_report()` and `package_versions()` from above.

**Acceptance criteria**

- A clean machine (fresh `uv sync`, or a new `venv` + install from the lock file) runs it with the documented commands.
- No secret is committed; `git log -p` shows only `.env.example`.
- Two runs with the same seed produce the same experiment output and the same report, apart from the timestamp.
- The README names at least one thing that pinning dependencies still does **not** guarantee.

**Checks to run yourself**

- Delete `.venv/`, recreate it from the lock file, run again — same result?
- Change the seed in `pyproject.toml` — does the output change *and* the report reflect the new value?
- Run on a second machine or in a container — which fields of the report differ, and do any of them matter for your experiment?

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Start from runtime_report() (section 1) and package_versions() (section 4).
#
# 1. Decide the files your capsule contains (see the project brief).
# 2. Implement export_environment_report(path), extending the facts below with:
#      - the installed version of each declared dependency
#      - the configured seed (read it from pyproject.toml [tool.experiment])
#      - a UTC timestamp
#    then write the result to `path` as JSON.
# 3. In the README, explain ONE limitation that remains after pinning
#    dependencies (native libraries? hardware? data? wall-clock nondeterminism?).

import json
from datetime import datetime, timezone


def export_environment_report(path: str) -> None:
    raise NotImplementedError("Implement the reproducible experiment capsule")


# export_environment_report("environment_report.json")
